**Lab type:** review  
**Course:** DS104 — Statistics for Data Science  
**Lesson:** Correlation: Measuring Relationships  
**Task:** An AI assistant was asked to analyse relationships in a customer survey dataset. Review its output: identify where it used the wrong correlation method, where it skipped visualisation, and where it used causal language for a correlational finding.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(7)

n = 800

# Customer survey data
# Overall satisfaction: ordinal 1-5 (integer)
overall_satisfaction = np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.20, 0.40, 0.25])

# Support quality: ordinal 1-5 (integer), correlated with overall_satisfaction
support_quality = np.clip(
    overall_satisfaction + np.random.choice([-1, 0, 0, 0, 1], n),
    1, 5
)

# Monthly spend (continuous), correlated with support quality (causation ambiguous)
monthly_spend = 50 + support_quality * 30 + np.random.normal(0, 40, n)
monthly_spend = np.clip(monthly_spend, 10, None)

# Months as customer (continuous), has a nonlinear relationship with spend
tenure_months = np.random.exponential(18, n).clip(1, 120).astype(int)
# Spend increases fast early, then levels off — nonlinear
spend_from_tenure = 20 * np.log1p(tenure_months) + np.random.normal(0, 15, n)

df = pd.DataFrame({
    'overall_satisfaction': overall_satisfaction,
    'support_quality': support_quality,
    'monthly_spend': monthly_spend.round(2),
    'tenure_months': tenure_months,
    'spend_from_tenure': spend_from_tenure.round(2)
})

print(df.dtypes)
print(df.head())

## AI-generated analysis

The AI was asked: *"Analyse the relationships between these variables and identify the strongest correlations."*

In [ ]:
# --- AI-GENERATED CODE ---

# Correlation matrix using Pearson (default df.corr())
corr_matrix = df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (Pearson)')
plt.show()

# AI conclusion (printed to simulate generated text)
print("=== AI Analysis Summary ===")
print(f"Overall satisfaction and support quality: r = {corr_matrix.loc['overall_satisfaction','support_quality']:.2f}")
print("→ High correlation confirms that improving support quality CAUSES higher satisfaction.")
print()
print(f"Support quality and monthly spend: r = {corr_matrix.loc['support_quality','monthly_spend']:.2f}")
print("→ Customers with better support experiences spend more. Strong relationship.")
print()
print(f"Tenure and spend (from tenure): r = {corr_matrix.loc['tenure_months','spend_from_tenure']:.2f}")
print("→ Weak correlation. Tenure does not predict spend.")

## Review questions

> **Question 1:** Two of the variables in this dataset are *ordinal* (integer scales 1–5). The AI used `df.corr()` which defaults to Pearson correlation. What is wrong with applying Pearson to ordinal data? Which correlation method should be used instead, and why?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Pearson assumes interval-scale continuous data** where equal differences between values are meaningful. On a 1–5 ordinal rating scale (like satisfaction or support quality), the gap between 1 and 2 may not be the same as the gap between 4 and 5 — respondents just rank, they don't measure. Applying Pearson to ordinal data can distort both the magnitude and direction of the relationship. **Spearman rank correlation** is the correct choice: it uses only rank order, making no assumptions about equal spacing, and remains valid for ordinal variables.

</details>

> **Question 2:** The AI concluded that *"improving support quality CAUSES higher satisfaction."* This uses causal language for a correlational finding. Describe two alternative explanations for the correlation between `support_quality` and `overall_satisfaction` that do not require causation in that direction.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Two alternative explanations for the support/satisfaction correlation:**

**Reverse causation:** Employees who are more satisfied overall tend to rate all aspects of their experience higher — including support. The causal arrow may run from general satisfaction to support ratings, not from support quality to satisfaction.

**Common cause (confounding):** Both support quality and overall satisfaction may be driven by a third variable, such as management effectiveness or team culture. A well-managed team creates both high-quality support interactions and higher job satisfaction, without one causing the other. Correlational data alone cannot distinguish these three scenarios.

</details>

> **Question 3:** The AI reported a weak Pearson r between `tenure_months` and `spend_from_tenure` and concluded that tenure does not predict spend. Before accepting this, what should you do? Run the code below to investigate.

In [ ]:
# Visualise the tenure vs spend relationship
plt.figure(figsize=(8, 5))
plt.scatter(df.tenure_months, df.spend_from_tenure, alpha=0.3, s=15)
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly spend from tenure')
plt.title('Tenure vs Spend — does the scatter plot tell a different story?')
plt.show()

# Also compute Spearman correlation
pearson_r, p_pearson = stats.pearsonr(df.tenure_months, df.spend_from_tenure)
spearman_r, p_spearman = stats.spearmanr(df.tenure_months, df.spend_from_tenure)
print(f'Pearson r = {pearson_r:.3f} (p={p_pearson:.4f})')
print(f'Spearman r = {spearman_r:.3f} (p={p_spearman:.4f})')

*(Write your answer: what does the scatter plot show? Why does Spearman give a different result from Pearson here?)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**The scatter plot reveals a nonlinear, threshold-style relationship:** spend rises steadily with tenure up to a point, then levels off — a pattern Pearson correlation, which only measures linear association, mostly misses. Pearson gives a low r because the overall trend is not a straight line. **Spearman measures monotonic rank association** — as long as spend consistently increases (or doesn't decrease) with tenure, Spearman captures it regardless of whether the relationship is linear or curved. This is why Spearman returns a meaningfully higher value here: the ranking relationship is strong even though the raw values don't follow a straight line.

</details>

## Step 2: Corrected analysis

Write a corrected correlation analysis for the ordinal variables using Spearman correlation, and restate the support/satisfaction finding with appropriate correlational language.

In [ ]:
# Corrected: use Spearman for ordinal columns
spearman_corr = df[['overall_satisfaction', 'support_quality']].corr(method='spearman')

rho, p = stats.spearmanr(df.overall_satisfaction, df.support_quality)
print(f'Spearman rho (satisfaction vs support): {rho:.3f}, p={p:.4f}')
print()
# Re-state the finding with correlational language
print('Corrected interpretation:')
print('Overall satisfaction and support quality are positively associated (ρ = {rho:.2f}).')
print('The direction and mechanism require further investigation —')
print('this finding does not establish that support quality causes satisfaction.')

## Wrap-up

> **Question 4:** Write a prompt you would use when handing the AI this dataset that prevents at least two of the three errors it made.

*(Write your prompt here.)*

<details>
<summary>🔑 Model prompt — Wrap-up</summary>

**Example strong prompt:**

> *Before running any correlation analysis, classify each variable as continuous, ordinal, or binary; use Spearman correlation for ordinal columns and point-biserial for binary; plot a scatter matrix to check for nonlinear patterns before interpreting any r values; report all findings as associations only — do not use causal language unless this data comes from a randomised experiment.*

**Why it's strong:** It prevents all three errors in this lab — wrong correlation method for ordinal data, missing the nonlinear pattern without visualisation, and causal overreach in the written conclusion.

</details>